# Alpine Crest Supervised Baseline And Threshold Tuning

Fit a small supervised baseline on Alpine Crest Private Bank feature-library outputs, then tune alert thresholds with capacity and cost tradeoffs. The model is a deterministic teaching baseline over tiny synthetic data, not a production model.


In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from banking_fraud_lab import (
    build_learner_facing_views,
    build_private_banking_features,
    evaluate_alert_scores,
    generate_private_banking_transaction_fraud_world,
)
from banking_fraud_lab.features import PRIVATE_BANKING_FEATURE_FAMILIES
from banking_fraud_lab.generators.private_banking import PRIVATE_BANKING_FALSE_POSITIVE_TYPE
from banking_fraud_lab.schema import PATTERN_IDS, PROTECTED_SCENARIO_ANSWER_KEYS

pd.set_option("display.max_columns", 40)


## Generate Learner-Facing Data

The supervised label comes from generated case outcomes. Protected scenario answer keys stay outside the learner-facing tables used for features and modeling.


In [ ]:
tables = generate_private_banking_transaction_fraud_world(
    seed=42,
    scenario_prevalence=0.2,
)
learner_tables = build_learner_facing_views(tables)

assert PROTECTED_SCENARIO_ANSWER_KEYS in tables
assert PROTECTED_SCENARIO_ANSWER_KEYS not in learner_tables

learner_summary = pd.DataFrame(
    {
        "table": ["cases", "case_outcomes", "alerts", "transactions"],
        "rows": [
            len(learner_tables["cases"]),
            len(learner_tables["case_outcomes"]),
            len(learner_tables["alerts"]),
            len(learner_tables["transactions"]),
        ],
    }
)
learner_summary


## Load Feature-Library Outputs

The model consumes `build_private_banking_features()` output. Feature-family metadata keeps inputs traceable to Detection pattern IDs.


In [ ]:
feature_frame = build_private_banking_features(learner_tables)
feature_family_map = pd.DataFrame(
    [
        {
            "family_id": spec.family_id,
            "detection_pattern_id": spec.detection_pattern_id,
            "output_columns": ", ".join(spec.output_columns),
        }
        for spec in PRIVATE_BANKING_FEATURE_FAMILIES
    ]
)
output_column_patterns = {
    output_column: {
        "feature_family": spec.family_id,
        "detection_pattern_id": spec.detection_pattern_id,
    }
    for spec in PRIVATE_BANKING_FEATURE_FAMILIES
    for output_column in spec.output_columns
}

assert set(feature_family_map["detection_pattern_id"]).issubset(set(PATTERN_IDS))
feature_family_map


## Build The Supervised Modeling Frame

Case outcomes provide labels after the Alert lifecycle has completed. The frame joins cases to feature-library outputs by transaction and Banking relationship.


In [ ]:
model_frame = (
    learner_tables["cases"][
        ["case_id", "alert_id", "transaction_id", "banking_relationship_id"]
    ]
    .merge(
        learner_tables["alerts"][["alert_id", "alert_type", "severity", "reason"]],
        on="alert_id",
        how="inner",
        validate="one_to_one",
    )
    .merge(
        learner_tables["case_outcomes"][
            ["case_id", "confirmed_fraud", "loss_amount_chf"]
        ],
        on="case_id",
        how="inner",
        validate="one_to_one",
    )
    .merge(
        feature_frame,
        on=["transaction_id", "banking_relationship_id"],
        how="inner",
        validate="one_to_one",
    )
)

assert model_frame["alert_id"].notna().all(), "model_frame.alert_id must be non-null"
assert model_frame["confirmed_fraud"].notna().all(), (
    "model_frame.confirmed_fraud must be non-null"
)
assert model_frame["alert_id"].is_unique
assert model_frame["confirmed_fraud"].nunique() == 2

model_frame[
    [
        "case_id",
        "alert_id",
        "alert_type",
        "confirmed_fraud",
        "amount_chf",
        "amount_to_aum_ratio",
        "is_new_counterparty",
        "is_cross_border",
        "rm_alert_share",
    ]
]


## Fit A Reproducible Baseline

The pipeline standardizes numeric feature-library outputs and fits a logistic-regression baseline. The fit uses tiny data to demonstrate mechanics, so interpretation should focus on feature direction and threshold behavior.


In [ ]:
feature_output_columns = [
    output_column
    for spec in PRIVATE_BANKING_FEATURE_FAMILIES
    for output_column in spec.output_columns
]
numeric_feature_columns = [
    column
    for column in feature_output_columns
    if pd.api.types.is_numeric_dtype(model_frame[column])
]
assert numeric_feature_columns

preprocess = ColumnTransformer(
    [("numeric", StandardScaler(), numeric_feature_columns)],
    remainder="drop",
)
baseline_model = Pipeline(
    [
        ("preprocess", preprocess),
        (
            "model",
            LogisticRegression(
                random_state=42,
                solver="lbfgs",
                max_iter=1000,
                class_weight="balanced",
            ),
        ),
    ]
)
target = model_frame["confirmed_fraud"].astype(bool)
baseline_model.fit(model_frame[numeric_feature_columns], target)

model_frame = model_frame.assign(
    score=baseline_model.predict_proba(model_frame[numeric_feature_columns])[:, 1].round(6)
)
alert_scores = model_frame[["alert_id", "score"]]
alert_scores


## Connect Coefficients To Detection Patterns

Coefficient direction is a compact way to inspect the tiny baseline. The feature-family mapping keeps each model input tied back to the Detection pattern vocabulary.


In [ ]:
coefficients = pd.DataFrame(
    {
        "feature": numeric_feature_columns,
        "coefficient": baseline_model.named_steps["model"].coef_[0],
    }
)
coefficients = coefficients.assign(
    feature_family=coefficients["feature"].map(
        lambda column: output_column_patterns[column]["feature_family"]
    ),
    detection_pattern_id=coefficients["feature"].map(
        lambda column: output_column_patterns[column]["detection_pattern_id"]
    ),
    absolute_coefficient=lambda frame: frame["coefficient"].abs(),
)
assert set(coefficients["detection_pattern_id"]).issubset(set(PATTERN_IDS))

coefficients.sort_values("absolute_coefficient", ascending=False).head(10)


## Tune Thresholds With Capacity And Costs

Thresholds are evaluated through the shared alert-aware utility. This keeps PR-AUC, precision, recall, alert volume, capacity utilization, and costs in the same report shape used elsewhere in the lab.


In [ ]:
report = evaluate_alert_scores(
    cases=model_frame[["case_id", "alert_id"]],
    case_outcomes=learner_tables["case_outcomes"],
    alert_scores=alert_scores,
    thresholds=(0.85, 0.75, 0.60, 0.40, 0.25),
    alert_capacity=2,
    investigation_cost_chf=250.0,
    false_positive_cost_chf=750.0,
    missed_fraud_cost_chf=50_000.0,
)
required_report_fields = {
    "pr_auc",
    "threshold_summaries",
    "cost_curve",
    "lowest_cost_threshold",
    "limitation_summary",
}
assert required_report_fields <= set(report)

threshold_summaries = pd.DataFrame(report["threshold_summaries"])
threshold_summaries[
    [
        "threshold",
        "precision",
        "recall",
        "alert_volume",
        "alert_capacity",
        "capacity_utilization",
        "over_capacity_alerts",
        "false_positives",
        "false_negatives",
        "total_cost_chf",
    ]
]


## Compare Cost Curve And Lowest-Cost Threshold

A high threshold can miss confirmed fraud, while a low threshold can create false positives and exceed review capacity. The lowest-cost threshold is selected by the evaluation utility from the supplied grid.


In [ ]:
cost_curve = pd.DataFrame(report["cost_curve"])
overview = pd.DataFrame(
    [
        {
            "pr_auc": report["pr_auc"],
            "lowest_cost_threshold": report["lowest_cost_threshold"],
            "case_count": report["population"]["case_count"],
            "confirmed_fraud_count": report["population"]["confirmed_fraud_count"],
        }
    ]
)

assert threshold_summaries["alert_capacity"].notna().all()
assert threshold_summaries["capacity_utilization"].notna().all()
assert (threshold_summaries["false_positives"] > 0).any()
assert (threshold_summaries["false_negatives"] > 0).any()

overview.merge(
    cost_curve,
    left_on="lowest_cost_threshold",
    right_on="threshold",
    how="left",
    validate="one_to_one",
)


## Inspect Legitimate High-Value False Positives

The generator includes Alpine Crest high-value activity that becomes a false-positive case outcome. These examples show why amount features need relationship context and threshold review instead of raw transaction amount alone.


In [ ]:
false_positive_examples = model_frame[
    model_frame["alert_type"].eq(PRIVATE_BANKING_FALSE_POSITIVE_TYPE)
    & ~model_frame["confirmed_fraud"].astype(bool)
].copy()
assert not false_positive_examples.empty

false_positive_examples[
    [
        "case_id",
        "alert_id",
        "alert_type",
        "amount_chf",
        "amount_to_aum_ratio",
        "amount_to_relationship_baseline_ratio",
        "is_new_counterparty",
        "is_cross_border",
        "score",
    ]
]


## Keep Evaluation Limits Visible

The report's limitation wording is part of the learner-facing output. It keeps the tiny supervised baseline focused on alert-capacity and cost behavior rather than headline accuracy.


In [ ]:
assert "accuracy is out of scope" in report["limitation_summary"]

pd.DataFrame(
    [
        {
            "pr_auc": report["pr_auc"],
            "lowest_cost_threshold": report["lowest_cost_threshold"],
            "limitation_summary": report["limitation_summary"],
        }
    ]
)
